In [1]:
import igraph as ig
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

rng = np.random.default_rng()

In [17]:
def simulate_uniform(
    graph: ig.Graph,
    window: float = 0,
    n_probs: int = 100,
    copies: int = 1000,
    low_p: float = 0.4,
    high_p: float = 0.6,
    no_bar: bool = False,
):
    n_edges = graph.ecount()
    lcc = np.zeros(n_probs, dtype="float32")
    all_prob = np.linspace(max(low_p, window), min(high_p, 1 - window), n_probs)
    for idx, p in tqdm(enumerate(all_prob), total=n_probs, disable=no_bar):
        is_removed = rng.random((copies, n_edges))
        link_prob = rng.uniform(p - window, p + window, (copies, n_edges))
        links_removed = is_removed >= link_prob
        for rep in range(copies):
            links_to_remove = np.flatnonzero(links_removed[rep])
            work = graph.copy()
            work.delete_edges(links_to_remove)
            lcc[idx] += max(work.components().sizes())
        lcc[idx] /= copies
    return all_prob, lcc / graph.vcount()


def multiple_run(graph: ig.Graph, n_probs: int, copies: int, n_win: int = 11):
    windows = np.linspace(0, 0.5, n_win)
    lcc, prob = [], []
    for win in tqdm(windows):
        temp_prob, temp_lcc = simulate_uniform(
            graph,
            window=win,
            n_probs=n_probs,
            copies=copies,
            low_p=0,
            high_p=1,
            no_bar=True,
        )
        lcc.append(temp_lcc)
        prob.append(temp_prob)
    return windows, prob, lcc

In [20]:
g = ig.Graph.Lattice([50, 50], circular=False)
windows, prob, lcc = multiple_run(g, 50, 20)
np.savez("uniform_grid", windows=windows, prob=prob, lcc=lcc)

  0%|          | 0/11 [00:00<?, ?it/s]

In [31]:
data = np.load("uniform_grid.npz")
(data["lcc"] == lcc).all()

True

In [4]:
# plt.figure(figsize=(8, 6))
%matplotlib qt
plt.axvline(0.5, c="k", zorder=50, lw=2)
n = len(windows)
colors = plt.cm.jet(np.linspace(0, 1, n))
for idx in range(n):
    plt.plot(prob[idx], lcc[idx], label=f"{windows[idx]*2:.2f}", c=colors[idx])
plt.legend(title="Window Size")
plt.title("Uniform Probability within a window Percolation")
plt.ylabel("Strength")
plt.xlabel("Middle Probability")
# plt.xlim(0.4, 0.6)

Text(0.5, 0, 'Middle Probability')